# Make a small 720p copy of the tactical video — resumable

**Runtime → Change runtime type → T4 GPU** (optional, makes it ~3× faster), then **Runtime → Run all**.
Accept the Google login prompts.

The video is encoded in 27 five-minute parts and each finished part is saved to
`My Drive/mtah/parts/`. **If Colab disconnects, just press Run all again** — finished parts are
skipped. At the end the parts are joined into `My Drive/mtah/BAR-ATH_tactical_720p.mp4`.
Every one of the 199,319 frames is kept, so frame *i* is still frame *i*.

In [ ]:
from google.colab import auth, drive
auth.authenticate_user()
drive.mount('/content/drive')

In [ ]:
import os, logging
logging.getLogger('googleapiclient').setLevel(logging.ERROR)
logging.getLogger('google_auth_httplib2').setLevel(logging.ERROR)
from googleapiclient.discovery import build
from googleapiclient.http import MediaIoBaseDownload
from googleapiclient.errors import HttpError
svc = build('drive', 'v3')
IDS = ['PASTE_TACTICAL_VIDEO_ID', 'PASTE_BROADCAST_VIDEO_ID']
files = {fid: svc.files().get(fileId=fid, fields='name,size', supportsAllDrives=True).execute() for fid in IDS}
for fid, m in files.items():
    print(m['name'], round(int(m.get('size', 0)) / 1e9, 2), 'GB')
TAC_ID = [fid for fid, m in files.items() if 'tactical' in m['name'].lower()][0]
SRC, SIZE = '/content/BAR-ATH_tactical.mp4', int(files[TAC_ID]['size'])

def download(fid):
    request = svc.files().get_media(fileId=fid, supportsAllDrives=True)
    with open(SRC, 'wb') as fh:
        dl = MediaIoBaseDownload(fh, request, chunksize=200 * 1024 * 1024)
        done = False
        while not done:
            status, done = dl.next_chunk()
            print('download %3d%%' % int(status.progress() * 100), end='\r')

def my_copy():
    """A copy of the tactical video that you own (so it has its own download quota).

    Uses a copy you made by hand in the Drive website ("Copy of BAR-ATH_tactical.mp4") if there
    is one; otherwise asks the API to make one, retrying because Google rate-limits copies of
    popular files.
    """
    import time
    for name in ('Copy of BAR-ATH_tactical.mp4', 'BAR-ATH_tactical_COPY.mp4'):
        found = svc.files().list(q=f"name = '{name}' and trashed = false",
                                 fields='files(id,size)').execute().get('files', [])
        for f in found:
            if int(f.get('size', 0)) == SIZE:
                print('using your copy:', name)
                return f['id']
    for attempt in range(6):
        try:
            print('asking Drive to copy the video into your Drive (attempt %d) ...' % (attempt + 1))
            new = svc.files().copy(fileId=TAC_ID, supportsAllDrives=True, fields='id',
                                   body={'name': 'BAR-ATH_tactical_COPY.mp4'}).execute()
            return new['id']
        except HttpError as e:
            if 'RateLimit' not in str(e) and 'rateLimit' not in str(e):
                raise
            time.sleep(30 * 2 ** attempt)
    raise SystemExit(
        'Drive will not copy the file through the API right now.\n'
        'Open the tactical video link in your browser, click the 3 dots -> "Make a copy",\n'
        'wait until "Copy of BAR-ATH_tactical.mp4" appears in My Drive, then run this cell again.')

if not (os.path.exists(SRC) and os.path.getsize(SRC) == SIZE):
    try:
        download(TAC_ID)
    except HttpError as e:
        if 'downloadQuotaExceeded' not in str(e):
            raise
        print('organiser file is over its download quota -> using a copy in your own Drive')
        download(my_copy())
print('\nsource ready:', round(os.path.getsize(SRC) / 1e9, 2), 'GB')

In [ ]:
import subprocess, time, math, shutil
TOTAL, FPS, PART = 199319, 25, 7500          # 7500 frames = 5 minutes per part
OUT = '/content/drive/MyDrive/mtah/parts'
os.makedirs(OUT, exist_ok=True)

gpu = subprocess.run('nvidia-smi', shell=True, capture_output=True).returncode == 0
nvenc = gpu and 'h264_nvenc' in subprocess.run('ffmpeg -hide_banner -encoders', shell=True,
                                               capture_output=True, text=True).stdout
ENC = (['-c:v', 'h264_nvenc', '-preset', 'p4', '-rc', 'vbr', '-cq', '30', '-b:v', '0', '-g', '50']
       if nvenc else ['-c:v', 'libx264', '-preset', 'veryfast', '-crf', '28', '-g', '50'])
print('encoder:', 'GPU (nvenc)' if nvenc else 'CPU (libx264)')

def frames_in(path):
    r = subprocess.run(['ffprobe', '-v', 'error', '-count_packets', '-select_streams', 'v:0',
                        '-show_entries', 'stream=nb_read_packets', '-of', 'csv=p=0', path],
                       capture_output=True, text=True)
    return int(r.stdout.strip() or 0)

n_parts = math.ceil(TOTAL / PART)
for k in range(n_parts):
    want = min(PART, TOTAL - k * PART)
    final = f'{OUT}/part_{k:03d}.mp4'
    if os.path.exists(final) and frames_in(final) == want:
        continue
    tmp = f'/content/part_{k:03d}.mp4'
    t0 = time.time()
    cmd = ['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-ss', str(k * PART / FPS),
           '-i', SRC, '-frames:v', str(want), '-vf', 'scale=1280:720', '-vsync', '0', '-an',
           *ENC, tmp]
    subprocess.run(cmd, check=True)
    got = frames_in(tmp)
    assert got == want, f'part {k}: {got} frames, expected {want}'
    shutil.move(tmp, final)
    print(f'part {k + 1}/{n_parts} done in {time.time() - t0:.0f}s')
print('all parts done')

In [ ]:
# Join the parts (no re-encoding) and check the total frame count.
parts = sorted(p for p in os.listdir(OUT) if p.endswith('.mp4'))
assert len(parts) == n_parts, f'{len(parts)} parts found, expected {n_parts} -- run the cell above again'
with open('/content/list.txt', 'w') as f:
    f.writelines(f"file '{OUT}/{p}'\n" for p in parts)
subprocess.run(['ffmpeg', '-hide_banner', '-loglevel', 'error', '-y', '-f', 'concat', '-safe', '0',
                '-i', '/content/list.txt', '-c', 'copy', '/content/BAR-ATH_tactical_720p.mp4'], check=True)
n = frames_in('/content/BAR-ATH_tactical_720p.mp4')
print('frames:', n, 'OK' if n == TOTAL else '<-- WRONG, tell Claude')
shutil.copy('/content/BAR-ATH_tactical_720p.mp4', '/content/drive/MyDrive/mtah/BAR-ATH_tactical_720p.mp4')
print('saved My Drive/mtah/BAR-ATH_tactical_720p.mp4,',
      round(os.path.getsize('/content/BAR-ATH_tactical_720p.mp4') / 1e9, 2), 'GB')